This is the main notebook for the Databricks YouTube Stats project.

I started by downloading the files from the Kaggle dataset to my local machine. Then importing those files to this Databricks project.

Now I want to create data frames for each CSV file individually, perform exploratory data analysis (EDA).

List of CSV files imported:
- CAvideos.csv
- DEvideos.csv
- FRvideos.csv
- GBvideos.csv
- INvideos.csv
- JPvideos.csv
- KRvideos.csv
- MXvideos.csv
- RUvideos.csv
- USvideos.csv

List of JSON files imported:
- CA_category_id.json
- DE_category_id.json
- FR_category_id.json
- GB_category_id.json
- IN_category_id.json
- JP_category_id.json
- KR_category_id.json
- MX_category_id.json
- RU_category_id.json
- US_category_id.json

In [0]:
# Import dependencies
import os
import json
from functools import reduce
from collections import Counter
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lower, lit, explode, to_date, to_timestamp, sum, size, log1p, count, when

In [0]:
# Load CAvideos.csv into a Spark DataFrame
CAvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/CAvideos.csv",
    header=True,
    inferSchema=True
)

display(CAvideos_df.limit(5))

In [0]:
# Load DEvideos.csv into a Spark DataFrame
DEvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/DEvideos.csv",
    header=True,
    inferSchema=True
)

display(DEvideos_df.limit(5))

In [0]:
# Load FRvideos.csv into a Spark DataFrame
FRvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/FRvideos.csv",
    header=True,
    inferSchema=True
)

display(FRvideos_df.limit(5))

In [0]:
# Load GBvideos.csv into a Spark DataFrame
GBvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/GBvideos.csv",
    header=True,
    inferSchema=True
)

display(GBvideos_df.limit(5))

In [0]:
# Load INvideos.csv into a Spark DataFrame
INvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/INvideos.csv",
    header=True,
    inferSchema=True
)

display(INvideos_df.limit(5))

In [0]:
# Load JPvideos.csv into a Spark DataFrame
JPvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/JPvideos.csv",
    header=True,
    inferSchema=True
)

display(JPvideos_df.limit(5))

In [0]:
# Load KRvideos.csv into a Spark DataFrame
KRvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/KRvideos.csv",
    header=True,
    inferSchema=True
)

display(KRvideos_df.limit(5))

In [0]:
# Load MXvideos.csv into a Spark DataFrame
MXvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/MXvideos.csv",
    header=True,
    inferSchema=True
)

display(MXvideos_df.limit(5))

In [0]:
# Load RUvideos.csv into a Spark DataFrame
RUvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/RUvideos.csv",
    header=True,
    inferSchema=True
)

display(RUvideos_df.limit(5))

In [0]:
# Load USvideos.csv into a Spark DataFrame
USvideos_df = spark.read.csv(
    "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/USvideos.csv",
    header=True,
    inferSchema=True
)

display(USvideos_df.limit(5))

Data Validation Goals
- Verify schema consistency across country datasets
- Check for missing/null values
- Detect duplicate records
- Validate numeric ranges and logical consistency
- Standardize schemas prior to dataframe union

Planned ETL Steps
- Validate raw datasets
- Add country metadata
- Union datasets into canonical dataframe
- Perform transformations/cleaning
- Generate analytical outputs and visualizations

Manually inspect schemas of all dfs. 

Later we'll try automation, which works better at scale.

In [0]:
# Manually inspect schemas
CAvideos_df.printSchema()

In [0]:
# Manually inspect schemas
DEvideos_df.printSchema()

In [0]:
# Manually inspect schemas
FRvideos_df.printSchema()

In [0]:
# Manually inspect schemas
GBvideos_df.printSchema()

In [0]:
# Manually inspect schemas
INvideos_df.printSchema()

In [0]:
# Manually inspect schemas
JPvideos_df.printSchema()

In [0]:
# Manually inspect schemas
KRvideos_df.printSchema()

In [0]:
# Manually inspect schemas
MXvideos_df.printSchema()

In [0]:
# Manually inspect schemas
RUvideos_df.printSchema()

In [0]:
# Manually inspect schemas
USvideos_df.printSchema()

In [0]:
# Manually inspect schemas
print(CAvideos_df.printSchema())
print(DEvideos_df.printSchema())
print(FRvideos_df.printSchema())
print(GBvideos_df.printSchema())
print(INvideos_df.printSchema())
print(JPvideos_df.printSchema())
print(KRvideos_df.printSchema())
print(MXvideos_df.printSchema())
print(RUvideos_df.printSchema())
print(USvideos_df.printSchema())

Manually verifying the schema, even for a limited number of dfs, is prone to error.

Let's try creating a function to verify the schemas.

In [0]:
# Create a dictionary of all dfs
country_dfs = {
    "CA": CAvideos_df,
    "DE": DEvideos_df,
    "FR": FRvideos_df,
    "GB": GBvideos_df,
    "IN": INvideos_df,
    "JP": JPvideos_df,
    "KR": KRvideos_df,
    "MX": MXvideos_df,
    "RU": RUvideos_df,
    "US": USvideos_df
}

In [0]:
# View the dictionary
for country, df in country_dfs.items():
    print(f"Country: {country}")
    print(df.printSchema())

This method still requires manual inspection. Let's try to automate.

In [0]:
# Start by making a list of expected columns
expected_columns = CAvideos_df.columns

# Then check if there are mismatches
for country, df in country_dfs.items():
    
    if df.columns != expected_columns:
        print(f"Schema mismatch detected in {country}")

In [0]:
# Let's check a different way, by seeing what differences exist
for country, df in country_dfs.items():

    missing = set(expected_columns) - set(df.columns)
    extra = set(df.columns) - set(expected_columns)

    print(f"{country}")
    print(f"Missing columns: {missing}")
    print(f"Extra columns: {extra}")

The automated verification shows the df schemas are identical.

In [0]:
# Check for null missing values in CAvideos_df
CAvideos_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in CAvideos_df.columns
]).display()

Checking for negative values showed some issues with the data, possibly messy CSV parsing, bad delimiter handling, or corrupted/misaligned columnns:

Check to see if views are negative (shouldn't be)

CAvideos_df.filter(col("views") < 0).display()

[CAST_INVALID_INPUT] The value ' Stephen Curry and Channing Tatum to Amy Schumer' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__lt__" was called from <command-8841710181262183>, line 2 in cell [36]



In [0]:
# Inpsect "views" column
CAvideos_df.select("views").show(20, truncate=False)

In [0]:
# Check how many rows are malformed
CAvideos_df.filter(~col("views").rlike("^[0-9]+$")).display(truncate=False)

It seems like the data is misaligned. Maybe a loading error.

Further sanity checks below.

In [0]:
# Check for null values
CAvideos_df.filter(col("video_id").isNull()).display()

There are no null values in "video_id" column, but still could be some ingestion issues.

Check for consistency again.

Reload a single CSV with stricter parsing.

In [0]:
# View a larger sample of CAvideos_df for manual inspection
# Looking for titles that look like descriptions, views with text, etc.
CAvideos_df.select("video_id", "title", "views").display(30, truncate=False)

Multi-line text fields are breaking the CSV row boundaries (likely)

Try a diagnosing with a fresh load.

In [0]:
# Load CAvideos.csv file again, allowing multilines, respect quote text blocks, handle internal quotes
# View a sample to see if it's handled correctly
df_raw = spark.read.option("header", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .csv("/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/CAvideos.csv")

df_raw.select("video_id", "title", "views").display()

In [0]:
#Check to see if views are negative (shouldn't be)
df_raw.filter(col("views") < 0).show(truncate=False)

In [0]:
# Count number of negative views
df_raw.filter(col("views") < 0).count()

The above shows there are zero rows were views are negative, a good sanity check.

In [0]:
# Check df_raw schema
df_raw.printSchema()

The schema was ingested by Spark as strings. Cast the necessasry columns to INT

In [0]:
# Cast columns to appropriate data types
df_typed = df_raw \
    .withColumn("views", col("views").cast("long")) \
    .withColumn("likes", col("likes").cast("long")) \
    .withColumn("dislikes", col("dislikes").cast("long")) \
    .withColumn("comment_count", col("comment_count").cast("long"))

df_typed.printSchema()

In [0]:
# View df
df_typed.display()

"trending_date" and "publish_time" are both clearly datetime columns

"trending_date" format: 17.15.11
"publish_time" format: 2017-11-13T15:48:57.000Z

These are different formats for different purposes. 

"trending_date" is a date.
"publish_time" is much more precise, it will require a timestamp.


In [0]:
# Cast "publish_time" to to_timestamp
df_typed = df_typed.withColumn(
    "publish_time",
    to_timestamp(col("publish_time"))
)

In [0]:
# Check data type of "publish_time"
df_typed.printSchema()

In [0]:
# View "publish_time" column
df_typed.select("publish_time").show(truncate=False)

In [0]:
# View "trending_date" column
df_typed.select("trending_date").show(truncate=False)

"trending_date" is in yy.dd.mm format

In [0]:
# Cast "trending_date" from string to to_date
df_typed = df_typed.withColumn(
    "trending_date",
    to_date(col("trending_date"), "yy.dd.MM")
)

In [0]:
# Check data type of "trending_date"
df_typed.printSchema()

In [0]:
# View "trending_date" column
df_typed.select("trending_date").show(truncate=False)

"trending_date" and "publish_time" have both been cast to appropriate data types.

Another method would be to create new columns, to avoid changing the data, then possibly removing the original columns if needed.

Check some columns ("video_error_or_removed", "ratings_disabled", "comments_disable") to see if they can be cast to Boolean.

In [0]:
# Check distinct entries of columns "video_error_or_removed"
df_typed.select("video_error_or_removed").distinct().show()


In [0]:
# Check distinct entries of columns "ratings_disabled"
df_typed.select("ratings_disabled").distinct().show()

In [0]:
# Check distinct entries of columns "comments_disabled"
df_typed.select("comments_disabled").distinct().show()

That method worked, but was not scalable.

Try to automate a little better.

In [0]:
# Create a list of the columns to test
possible_boolean_cols = ["video_error_or_removed", "ratings_disabled", "comments_disabled"]

In [0]:
# Create a loop to view them all at the same time
for c in possible_boolean_cols:
    df_typed.groupBy(c).count().show()

A function would make this process repeatable, better for pipeline workflows.

In [0]:
# Create profile_columns function for reusability
def profile_columns(df, cols):
    for c in cols:
        print(f"\n=== {c} ===")
        df.groupBy(c).count().orderBy("count", ascending=False).show()

In [0]:
# Call the function
profile_columns(df_typed, possible_boolean_cols)


After running the profile_columns function, it's clear that the columns are binary and can safely be cast to Boolean.

In [0]:
# Check data types again
df_typed.printSchema()

In [0]:
# Safely cast to Boolean
df_typed = df_typed \
    .withColumn("comments_disabled", lower(col("comments_disabled")).cast("boolean")) \
    .withColumn("ratings_disabled", lower(col("ratings_disabled")).cast("boolean")) \
    .withColumn("video_error_or_removed", lower(col("video_error_or_removed")).cast("boolean"))

In [0]:
# Check data types to verfiy cating
df_typed.printSchema()

"category_id" column should be examined.

In [0]:
# Examine "category_id" column
df_typed.groupBy("category_id").count().show()

"CA_category_id.json" likely has information about this column. Load into a df

In [0]:
# Load "CA_category_id.json" into a df
df_categories = spark.read.option("multiLine", True).json("/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/CA_category_id.json")

In [0]:
# View df_categories
df_categories.printSchema()

In [0]:
# Verify what we have
row = df_categories.select("items").limit(1).collect()[0]
print(type(row))
print(row)

In [0]:
# View nested items
row = df_categories.select("items").limit(1).collect()[0].asDict(recursive=True)
print(json.dumps(row, indent=2))

Finally, a view of the nested strucuture of the category id JSON file. 

Next let's check for null top level-fields

In [0]:
# Check for null
df_categories.select(
    col("etag").isNull().alias("etag_null"),
    col("kind").isNull().alias("kind_null"),
    col("items").isNull().alias("items_null")
).show()

In [0]:
# Check the size of the JSON df
df_categories.select(size("items").alias("item_count")).show()

Next, turn array into rows, flatten and then finally join.

Was enough verification, exploration done?

In [0]:
# Turn arrays into rows
df_exploded = df_categories.selectExpr("explode(items) as item")

In [0]:
# Flatten the nested structure
df_lookup = df_exploded.select(
    col("item.id").alias("category_id"),
    col("item.snippet.title").alias("category_name")
)

In [0]:
# Join the dfs (df_typed, df_lookup)
df_final = df_typed.join(df_lookup, on="category_id", how="left")

In [0]:
# View a small sample of the joined df
df_final.display(20, truncate=False)

In [0]:
# Double check the join worked
df_final.select("category_id", "category_name").distinct().show()

In case we create a large df for the complete dataset, a region column should be created for each CSV/df

In [0]:
# Create a region column for entire df
df_final = df_final.withColumn("region", lit("CA"))

In [0]:
df_final.display()

This df is silver by now, to make it gold remove some unnecesary columns

In [0]:
# Create the final columns for the df
df_gold = df_final.select(
    "video_id",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "region"
)

In [0]:
# Add a like_ratio column
df_gold = df_gold.withColumn(
    "like_ratio",
    col("likes") / col("views")
)

In [0]:
# Displap df_gold
df_gold.display()

In [0]:
# Rearrange some of the columns for readability
df_gold = df_gold.select(
    "video_id",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "like_ratio",
    "comment_count",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "region"
)

In [0]:
df_gold.display()

Do some validation checks, perhaps should have been done before join?

Read ETL PDF from Databricks. Might make sense to do validation checks before and after? Maybe just after?

Check all columns for null

Check numeric values for impossible values (did some already when checked "views" column for negataive)

Likes should not exceed views
Dislikes should not exceed views
Comments should not exceed views

Trending date should be after publishing date

Check for duplicates

Check profiling describe(), to expose strange mac values or unexpected distributions,

In [0]:
# Check all columns for null values
df_gold.select([
    count(when(col(c).isNull(), c)). alias(c)
    for c in df_gold.columns
]).show()

There are 74 null values in category_name, other columns have zero.

In [0]:
# Check numeric columns for impossible values
# Create list of numeric values
numeric_cols = [
    "views",
    "likes",
    "dislikes",
    "comment_count"
]

# Loop through numeric columns to check for negative numbers
for c in numeric_cols:
    count_neg = df_gold.filter(col(c) < 0).count()

    print(f"{c}: {count_neg}")

There are no negative numeric values or impossible numeric values.

Check for logical consistency.

In [0]:
# Likes should not exceed views
df_typed.filter(
    col("likes") > col("views")
).count()

In [0]:
# Dislikes should not exceed views
df_typed.filter(
    col("dislikes") > col("views")
).count()

In [0]:
# Comments should not exceed views
df_typed.filter(
    col("comment_count") > col("views")
).count()

In [0]:
# Publish date after trending date?
df_typed.filter(
    col("publish_time") > col("trending_date")
).count()

In [0]:
# Check number of rows in df_gold
df_gold.count()

In [0]:
# Inspect the gold table
df_gold.select(
    "trending_date",
    "publish_time"
).show(10, truncate=False)

In [0]:
# Inspect the df_raw times
df_raw.select(
    "trending_date",
    "publish_time"
).show(10, truncate=False)

Error in formatting, should be yy.dd.MM not yy.dd.mm (MM for months, mm for minutes).

This issue was corrected in Cell 56.

Still an issue with 2031 rows containing a publish time greater than the trending time.

In [0]:
# Look at some of the bad rows
df_gold.filter(col("publish_time") > col("trending_date")) \
    .select("video_id", "publish_time", "trending_date") \
    .show(20, truncate=False)

It looks like Spark is handling the "trending_date" as 00:00:00, so anything later than that on the same day in "publish_time" is greater.

Validation needs to be more precise.


In [0]:
# Publish date after trending date?
df_gold.filter(
    to_date(col("publish_time")) > col("trending_date")
).count()

Excellent, there are no longer any publish times greater than trending dates. The sanity check is successful.

Check for duplicates.

In [0]:
# Check for duplicate rows
df_gold.groupBy(
    "video_id",
    "trending_date",
    "region"
).count().filter(
    col("count") > 1
).show()

There are no duplicate rows.

Check that the category names all mapped properly.

In [0]:
# Check for null category names
df_gold.filter(
    col("category_name").isNull()
).count()

We have 74 category_names that are null, but we did a null check before and there were zero?

In [0]:
# Inspect null rows
df_final.filter(col("category_name").isNull()) \
    .select("category_id") \
    .distinct() \
    .show()

In [0]:
# View df_gold, manually inspect nulls
df_gold.display()

In [0]:
# View df_final, manually inspect nulls
df_final.display()

Manual inspection showed that category 29 is the null values and the the US JSON ("US_category_id.json") contains this category.

Manual inspection is not scalable or precise, a better solution is to flatten all JSONs, union them, create a global table, and resolve conflicts.

Write a function to load all JSONs

In [0]:
# Function to load a category JSON and add region
def load_category_json(region):
    
    file_path = f"/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/{region}_category_id.json"

    return (
        spark.read
        .option("multiline", "true")
        .json(file_path)
        .withColumn("region", lit(region))
    )

In [0]:
# List of regions
regions = [
    "CA",
    "DE",
    "FR",
    "GB",
    "IN",
    "JP",
    "KR",
    "MX",
    "RU",
    "US"
]

In [0]:
# Load all JSON files

# Create empty category_dfs list
category_dfs = []

# Loop through regions while calling new load_category_json function
for region in regions:
    category_dfs.append(
        load_category_json(region)
    )

Forget to add a region row for each JSON, have to run, last few cells done in haste, need to double check.

In [0]:
# Create one DataFrame from all JSON files
from functools import reduce

df_categories_all = reduce(
    lambda x, y: x.unionByName(y),
    category_dfs
)

Check that region has been added.

In [0]:
# View all the columns of df_categories_all
df_categories_all.columns

'_corrupt_record' is not a good sign, something is off.

Reload a single JSON to try and debug.

This was resolved, the previous cell now lists expected columns.

In [0]:
# Read US JSON on its own
df_test = spark.read.option(
    "multiline", "true"
).json("/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/US_category_id.json")

In [0]:
# View test schema
df_test.printSchema()

The last two cells were tests to ensure JSONs were loading properly. This issue has been debugged/troubleshooted.

Next we need to flatten the JSONs to help with a join.

In [0]:
# Flatten the JSON
df_categories_flat = (
    df_categories_all
    .select(
        explode("items").alias("item"),
        col("region")
    )
    .select(
        col("item.id").alias("category_id"),
        col("item.snippet.title").alias("category_name"),
        col("region")
    )
)

In [0]:
# View the df_categories_flat
df_categories_flat.display()

A decision to make, whether to reduce this into an analytical list of all categories or maintain a raw/reference file that maintains regions.

A deduplicated category df should be created for analysis, while preserving the region-level reference data separately.


Something I should have done previously, check that files loaded properly.

Run a loop to see if there are any obvious issues.

In [0]:
# View which files loaded correctly
for i, df in enumerate(category_dfs):
    print(f"DF {i}")
    df.printSchema()
    print("Rows:", df.count())
    print("-" * 30)

Create the deduplicated df by selecting only the "category_id" and "category_name"

In [0]:
# Create the deduplicated
df_category_dim = (
    df_categories_flat
    .select("category_id", "category_name")
    .distinct()
)

In [0]:
# View the new  df_category_dim
df_category_dim.display()

Create a sorted df, casting the category_id to INT

In [0]:
# Create the deduplicated df with more sorting
df_category_dim = (
    df_categories_flat
    .select(
        col("category_id").cast("int"),
        "category_name"
    )
    .distinct()
    .orderBy("category_id")
)

In [0]:
# View the new  df_category_dim
df_category_dim.display()

In [0]:
# Join the dfs (df_typed, df_category_dim)
df_CA_joined = df_final.join(df_category_dim, on="category_id", how="left")

In [0]:
# Check all columns for null values
df_CA_joined.select([
    count(when(col(c).isNull(), c)). alias(c)
    for c in df_CA_joined.columns
]).display()

In [0]:
# Check columns df_typed
df_final.columns

In [0]:
# Check columns df_gold
df_gold.columns

df_typed has "category_id" column, df_gold does not have "category_id"

There are too many dfs floating around is this convoluted exploration notebook. Polised ETL book should have a much more efficient pipeline.

In [0]:
# Drop "category_name" from df_type
df_typed = df_typed.drop("category_name")

# Print schema to verify
df_typed.printSchema()

In [0]:
# Try the join again with removed "category_name"
# Check all columns for null values
df_CA_joined.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_CA_joined.columns
]).display()

Issue persists...

In [0]:
# Count number of "category_name" columns
Counter(df_gold.columns)

In [0]:
# Count number of "category_name" columns
Counter(df_CA_joined.columns)

In [0]:
# Count number of "category_name" columns
Counter(df_typed.columns)

In [0]:
# Count number of "category_name" columns
Counter(df_final.columns)

Let's try a different join, and a new df name

In [0]:
# Join the dfs (df_typed, df_category_dim)
df_CA_cat_dim = df_typed.join(df_category_dim, on="category_id", how="left")

In [0]:
# Check all columns for null values
df_CA_cat_dim.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_CA_cat_dim.columns
]).display()

Looks like the errors were human confusion, too many different DataFrames with different names and purposes.

df_typed was the relevant DataFrame (cleaned, casted) to be joined with the "df_category_dim" analytic category DataFrame.

Now we have a new problem, the description column suddenly has 1296 null values. Check df_typed description column for null values.

In [0]:
# Check df_typed for null values
df_typed.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_typed.columns
]).display()


In [0]:
# Check df_gold for null values
df_gold.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_gold.columns
]).display()

In [0]:
# Check initial df_raw for null values
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).display()

`description` column, and others, is not required for analytic purposes. The column was previously dropped in a former version of `df_gold`.

Recreate `df_gold` form new `df_CA_cat_dim`

In [0]:
# Create the final columns for the df
df_gold = df_CA_cat_dim.select(
    "video_id",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "region"
)

In [0]:
# View new df_gold
df_gold.display()

Disorganized notebook, having to redo work, adding a region column now.

This is a tricky move, in that the JSON file contains `category_id` not from the CA JSON file.

*Note: Polished notebook should handle this differently.

In [0]:
# Create a region column for entire df
df_CA_cat_dim = df_CA_cat_dim.withColumn("region", lit("CA"))

In [0]:
# Create the final columns for the df
df_gold = df_CA_cat_dim.select(
    "video_id",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",
    "region"
)

In [0]:
# View new df_gold
df_gold.display()

In [0]:
# Check df_gold for null values
df_gold.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_gold.columns
]).display()

In [0]:
# Publish date after trending date?
df_gold.filter(
    to_date(col("publish_time")) > col("trending_date")
).count()

In [0]:
# Check for duplicate rows
df_gold.groupBy(
    "video_id",
    "trending_date",
    "region"
).count().filter(
    col("count") > 1
).show()

In [0]:
# Check for null category names
df_gold.filter(
    col("category_name").isNull()
).count()

In [0]:
# Print Schema of df_gold
df_gold.printSchema()

In [0]:
# Add a like_ratio column
df_gold = df_gold.withColumn(
    "like_ratio",
    col("likes") / col("views")
)

In [0]:
# Print Schema
df_gold.printSchema()

In [0]:
# Check for impossible negative values
for c in numeric_cols:
    count_neg = df_gold.filter(col(c) < 0).count()

    print(f"{c}: {count_neg}")

In [0]:
# Likes should not exceed views
df_typed.filter(
    col("likes") > col("views")
).count()

In [0]:
# Dislikes should not exceed views
df_typed.filter(
    col("dislikes") > col("views")
).count()

In [0]:
# Comments should not exceed views
df_typed.filter(
    col("comment_count") > col("views")
).count()

In [0]:
# Profiling numeric columns
df_gold.describe(
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "like_ratio"
).show()

OK, this is a messy exploration notebook but we are getting a bit more cleaned up now.

There's a clean df_gold, with multiple validation steps, that we can progress with from here. This only contains Canada video CSV data, but JSON category ID data from every region.

A larger DataFrame with all countries could be created.

First let's look at possibly adding some more Business Metrics:
- Engagement rate
- Comment rate
- Dislike ratio
- Net engagement score (like - dislikes) + commment_count
- Virality indicator
- Log scale
- Engagament quality (weighted score)

For now maybe just: engagement_rate, comment_rate, log_views


In [0]:
# Add an engagement_rate column
df_gold = df_gold.withColumn(
    "engagement_rate",
    (col("likes") + col("dislikes") + col("comment_count")) / col("views")
)

In [0]:
# Add a comment_rate column
df_gold = df_gold.withColumn(
    "comment_rate",
    col("comment_count") / col("views")
)

In [0]:
# Add a log_views column
df_gold = df_gold.withColumn(
    "log_views",
    log1p(col("views"))
)

In [0]:
# View new df_gold
df_gold.display()

In [0]:
# Print Schema of df_gold
df_gold.printSchema()

In [0]:
# Rearrange columns for a good mental grouping
df_gold = df_gold.select(
    "video_id",
    "region",
    "title",
    "channel_title",
    "category_name",
    "publish_time",
    "trending_date",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "like_ratio",
    "engagement_rate",
    "comment_rate",
    "log_views",
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed",    
)

df_gold.display()

At this point we have a rough proof of concept before creating a large DataFrame for the entire dataset.

Data has been ingested, examined, casted, validated, filtered, joined. The process can work.

Let's try designing some functions to automate some of these processes and create the large DataFrame

In [0]:
# Create a set of allowed regions
ALLOWED_REGIONS = {
    "CA", "DE", "FR", "GB", "IN",
    "JP", "KR", "MX", "RU", "US"
}

In [0]:
# Create the bronze ingestion function
def load_raw_csv(spark, path: str, region: str):
    
    # Validate region (fail fast)
    if region not in ALLOWED_REGIONS:
        raise ValueError(f"[INGESTION BLOCKED] Unknown region: {region}")

    # Read raw file
    df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", "\"")
    .option("inferSchema", True)
    .csv(path)
    )

    # Add lineage column
    df = df.withColumn("region", lit(region))

    return df

In [0]:
def load_all_regions(spark, folder_path: str):
    
    dfs = []

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            region = file.split("videos.csv")[0]
            print(f"Loading: {file} → region: {region}")  # ← add this

            df = load_raw_csv(
                spark,
                os.path.join(folder_path, file),
                region
            )

            dfs.append(df)

    print(f"Total DataFrames loaded: {len(dfs)}")  # ← and this

    if not dfs:
        raise ValueError(f"[INGESTION FAILED] No CSV files found in: {folder_path}")

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

In [0]:
# After union, verify no rows were lost
def validate_bronze(df, expected_regions: set):

    print("=== BASIC VALIDATION ===")

    required_cols = ["video_id", "views", "likes", "region"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "_corrupt_record" in df.columns:
        bad = df.filter(col("_corrupt_record").isNotNull()).count()
        print(f"Corrupt records: {bad}")

    print("Row count:", df.count())

    # Reconciliation: confirm all expected regions loaded
    loaded_regions = {r.region for r in df.select("region").distinct().collect()}
    missing_regions = expected_regions - loaded_regions
    if missing_regions:
        print(f"[WARNING] Missing regions after union: {missing_regions}")
    else:
        print(f"[OK] All {len(loaded_regions)} regions present")

    df.select("region").distinct().show()

    return df

In [0]:
# Bronze to silver transformation: cleaning and casting
# Expects raw string columns from bronze layer: do not call twice.
def transform_silver(df):
    return (
        df
        .withColumn("publish_time", to_timestamp("publish_time"))
        .withColumn("trending_date", to_date("trending_date", "yy.dd.MM"))
        .withColumn("views", col("views").cast("long"))
        .withColumn("likes", col("likes").cast("long"))
        .withColumn("dislikes", col("dislikes").cast("long"))
        .withColumn("comment_count", col("comment_count").cast("long"))
        .select(
            "video_id",
            "category_id",
            "title",
            "channel_title",
            "publish_time",
            "trending_date",
            "views",
            "likes",
            "dislikes",
            "comment_count",
            "comments_disabled",
            "ratings_disabled",
            "video_error_or_removed",
            "region"
        )
    )

In [0]:
# Orchestration layer
folder_path = "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/"

df_bronze = load_all_regions(spark, folder_path)
df_bronze = validate_bronze(df_bronze, expected_regions=ALLOWED_REGIONS)
df_silver = transform_silver(df_bronze)

In [0]:
# Load a single category JSON file
def load_raw_json(spark, file_path: str, region: str):

    df = (
        spark.read
             .option("multiline", "true")
             .json(file_path)
    )

    if df.limit(1).count() == 0:
        raise ValueError(f"[INGESTION FAILED] File is empty or unreadable: {file_path}")

    df = df.withColumn("region", lit(region))

    return df

In [0]:
# Create a batch loader for all category JSONs
def load_all_category_jsons(spark, folder_path: str):

    dfs = {}

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            region = file.split("_category_id")[0]

            df = load_raw_json(
                spark,
                os.path.join(folder_path, file),
                region
            )

            dfs[region] = df

    if not dfs:
        raise ValueError(f"[INGESTION FAILED] No JSON files found in: {folder_path}")

    return dfs

In [0]:
# Create function to validate raw category JSONs
def validate_category_raw(dfs: dict) -> dict:
    """
    Schema sanity check across all region DataFrames.
    Logs mismatches and row counts, then returns the dfs dict.
    """

    if not dfs:
        raise ValueError("[VALIDATION FAILED] Empty dataframe dictionary passed.")

    # Use simpleString() for reliable schema comparison
    base_region, base_df = next(iter(dfs.items()))
    base_schema_str = base_df.schema.simpleString()

    valid_dfs = {}

    for region, df in dfs.items():
        schema_str = df.schema.simpleString()

        if schema_str != base_schema_str:
            print(f"[WARNING] Schema mismatch in {region} vs {base_region} — skipping")
            print(f"  Expected: {base_schema_str}")
            print(f"  Got:      {schema_str}")
            continue

        row_count = df.count()
        print(f"[OK] {region}: rows={row_count}")
        valid_dfs[region] = df

    if not valid_dfs:
        raise ValueError("[VALIDATION FAILED] No DataFrames passed schema validation.")

    return valid_dfs  

In [0]:
def flatten_categories(df):
    """Explode items array and select relevant fields."""

    return (
        df
        .select(explode("items").alias("item"), "region")
        .select(
            col("item.id").alias("category_id"),
            col("item.snippet.title").alias("category_name"),
            "region"
        )
    )

In [0]:
# Orchestration: creating necesary variables
folder_path = "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/"

category_raw_dfs   = load_all_category_jsons(spark, folder_path)
category_valid_dfs = validate_category_raw(category_raw_dfs)

Create a union of the validated JSON dfs from validate_category raw. Save a raw df for reference, but create a caregory id lookup table that is deduplicated.

In [0]:
# Union all region category DataFrames into one
category_unioned_df = reduce(DataFrame.unionByName, category_valid_dfs.values())

# Flatten: explode items array, extract category_id and category_name
category_flattened_df = flatten_categories(category_unioned_df)

# Keep raw union for reference
category_raw_union_df = category_flattened_df

# Deduplicate on category_id to create the dimension table
category_dim_df = (
    category_flattened_df
    .dropDuplicates(["category_id"])
    .drop("region")
    .orderBy("category_id")
)

Now finally create a large df_gold, a join of all the files in the dataset.

In [0]:
# A cleaned and validated DataFrame containing all the relevant fields from the dataset
df_gold = df_silver.join(category_dim_df, on="category_id", how="left")

In [0]:
# View df_gold
df_gold.display()

In [0]:
# View schema
df_gold.printSchema()

`df_gold` is looking like it's in a good place.

Create a function to add the business metrics from previous iterations in this exploration notebook

In [0]:
# Creat a function to add some business metrics: engagement_rate, comment_rate, log_views
def add_business_metrics(df):

    return (
        df
        .withColumn(
            "engagement_rate",
            when(
                col("views") > 0,
                (
                    col("likes")
                    + col("dislikes")
                    + col("comment_count")
                ) / col("views")
            )
        )
        .withColumn(
            "comment_rate",
            when(
                col("views") > 0,
                col("comment_count") / col("views")
            )
        )
        .withColumn(
            "log_views",
            log1p(col("views"))
        )
    )

In [0]:
# Call add_business_metrics function
df_gold = add_business_metrics(df_gold)
# View df_gold
df_gold.display()
# View schema
df_gold.printSchema()

Cells below show failed attempt to load and union JSON files. Can be ignored, should possibly be removed. They remain to see the process and remember possible pitfalls, clean notebook should not contain these files. 

In [0]:
# Flatten dfs
category_flat_dfs  = {
    region: flatten_categories(df)
    for region, df in category_valid_dfs.items()
}

In [0]:
# Union dfs
df_category_dim = reduce(
    lambda a, b: a.unionByName(b),
    category_flat_dfs.values()
)

Issues with functions not running, problem from `reduce` importing. 

Imported reduce explicitly from `from pyspark.sql.functions import reduce` and `from pyspark.sql.functions import *`

That conflicted with `from functools import reduce`, Spark overwrites that with its own `reduce`

Following few cells are part of debugging that error

In [0]:

print(os.listdir("/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/"))

In [0]:
folder_path = "/Workspace/Users/nickbartram25@gmail.com/youtube_stats_databricks/Resources/"

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        region = file.split("videos.csv")[0]
        print(f"file: {file} → region: '{region}' → allowed: {region in ALLOWED_REGIONS}")

In [0]:
print(reduce)

In [0]:
def load_all_regions_debug(spark, folder_path: str):
    
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            region = file.split("videos.csv")[0]

            df = load_raw_csv(
                spark,
                os.path.join(folder_path, file),
                region
            )

            print(f"\n{region}: {df.columns}")

load_all_regions_debug(spark, folder_path)

In [0]:
import traceback

try:
    df_bronze = load_all_regions(spark, folder_path)
except Exception as e:
    traceback.print_exc()

Cells above show failed attempt to load and union JSON files. Can be ignored, should possibly be removed. They remain to see the process and remember possible pitfalls, clean notebook should not contain these files. 

For next time:

- JOIN large CSV file with large JSON file
- Add Business Metrics
- Check for extra validation steps
- Consider creating a polished ETL_notebook
- Consider visualizations